# Spatial CV · LightGBM quantile regression

Обучение p10, p50, p90 на 4 пространственных фолдах внутри городов. Предсказания test усредняются по фолдам. Метрика — официальная `data/metric_cv.py`. Выполнять из корня репозитория (`Restart & Run All`).

**Примечание:** выбранные географические блоки ~3 км — собственный подход к CV; это не гарантированно тот же способ, что в baseline преподавателя.

In [ ]:
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

SEED = 42
N_FOLDS = 4
QUANTILES = (0.1, 0.5, 0.9)
MODEL_PARAMS = dict(
    objective="quantile", n_estimators=180, learning_rate=0.055,
    num_leaves=15, min_child_samples=30, colsample_bytree=0.95,
    random_state=SEED, n_jobs=2, verbosity=-1,
)
started = time.monotonic()
tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
assert "daily_orders" in tr and "daily_orders" not in te
assert tr["id"].is_unique and te["id"].is_unique
assert te["id"].tolist() == sample["id"].tolist()
print("train/test:", tr.shape, te.shape)

## Признаки и географические группы

Категориальные признаки кодируем как `category`. Группы формируем из координат внутри каждого города. `GroupKFold` не смешивает один геоблок между train и validation.

In [ ]:
CATS = ["city", "district_type", "entrance_type"]
feature_cols = [c for c in te.columns if c != "id"]
X = tr[feature_cols].copy()
X_test = te[feature_cols].copy()
for col in CATS:
    categories = pd.Index(pd.concat([X[col], X_test[col]], ignore_index=True).dropna().unique())
    X[col] = pd.Categorical(X[col], categories=categories)
    X_test[col] = pd.Categorical(X_test[col], categories=categories)
y = tr["daily_orders"].to_numpy(float)

lat_step = 3.0 / 111.0
lat_rad = np.deg2rad(tr["latitude"].to_numpy(float))
lon_step = 3.0 / (111.0 * np.maximum(np.cos(lat_rad), 0.2))
lat_bin = np.floor(tr["latitude"].to_numpy(float) / lat_step).astype(int)
lon_bin = np.floor(tr["longitude"].to_numpy(float) / lon_step).astype(int)
groups = (tr["city"].astype(str) + ":" + pd.Series(lat_bin).astype(str)
          + ":" + pd.Series(lon_bin).astype(str)).to_numpy()
print("Geoblocks:", len(np.unique(groups)))
splits = list(GroupKFold(n_splits=N_FOLDS).split(X, y, groups=groups))
for k, (a, b) in enumerate(splits):
    assert set(groups[a]).isdisjoint(set(groups[b]))
    print("fold", k, "train", len(a), "valid", len(b))

## OOF и ансамбль на test

Каждая строка train получает OOF-предсказание от модели, которая не обучалась на этой строке. Квантили упорядочиваем одинаково на validation и test.

In [ ]:
def postprocess(pred):
    return np.round(np.clip(np.sort(np.asarray(pred, dtype=float), axis=1), 0, 5000), 1)

oof = np.full((len(tr), 3), np.nan)
test_sum = np.zeros((len(te), 3), dtype=float)
qualities = []
for fold, (a, b) in enumerate(splits):
    valid = np.empty((len(b), 3), dtype=float)
    test_pred_fold = np.empty((len(te), 3), dtype=float)
    for j, q in enumerate(QUANTILES):
        model = LGBMRegressor(alpha=q, **MODEL_PARAMS)
        model.fit(X.iloc[a], y[a], categorical_feature=CATS)
        valid[:, j] = model.predict(X.iloc[b])
        test_pred_fold[:, j] = model.predict(X_test)
    valid = postprocess(valid)
    oof[b] = valid
    test_sum += test_pred_fold / N_FOLDS
    quality = quantile_quality(
        y[b], valid[:, 0], valid[:, 1], valid[:, 2],
        tr["city"].iloc[b]
    )
    qualities.append(quality)
    print(f"fold {fold}: 100*P={100*quality:.2f}; elapsed={time.monotonic()-started:.1f}s")
assert np.isfinite(oof).all()
print("Mean 100*P:", round(100*np.mean(qualities), 2))

## Submission и cv.json

Создаём файлы из фактического запуска. `submission.csv` должен совпадать с `sample_submission.csv` по строкам и колонкам. В `cv.json` сохраняется официальная метрика `100·P` по фолдам.

In [ ]:
test_pred = postprocess(test_sum)
sub = sample.copy()
sub.iloc[:, 1:] = test_pred
assert list(sub.columns) == [
    "id", "predicted_daily_orders_p10",
    "predicted_daily_orders_p50", "predicted_daily_orders_p90"
]
assert len(sub) == len(te) and sub["id"].tolist() == te["id"].tolist()
values = sub.iloc[:, 1:].to_numpy(dtype=float)
assert np.isfinite(values).all()
assert ((values[:, 0] >= 0) & (values[:, 0] <= values[:, 1]) &
        (values[:, 1] <= values[:, 2]) & (values[:, 2] <= 5000)).all()
sub.to_csv("submission.csv", index=False)
cv = cv_json("lgb_spatial_3km_4fold", qualities)
print(f"Rows={len(sub)}; elapsed={time.monotonic()-started:.1f}s")

In [ ]:
print(json.dumps(cv, ensure_ascii=False))